# 10: Streaming and Real-Time Processing

## Objective
Implement IQ streaming, on-FPGA averaging, and real-time decimation.

In [ ]:
BITSTREAM_PATH = "/path/to/firmware.bit"
USE_PROXY = False
PROXY_IP = "192.168.1.100"

from qick import QickSoc, QickConfig
import numpy as np
import matplotlib.pyplot as plt

if USE_PROXY:
    from qick.pyro import make_proxy
    soc = make_proxy(PROXY_IP)
else:
    soc = QickSoc(bitfile=BITSTREAM_PATH)

soc.reset_gens()
soc.reset_adcs()
print("Ready")

In [ ]:
config = QickConfig(reps=1, soft_avgs=1)

config.stream_enable(True)
config.stream_channel(ch=0, decimation=10)

config.pulse(ch=0, style="const", length=1000, gain=0.5)

soc.run(config)
stream_data = soc.get_stream()

plt.figure()
plt.plot(stream_data['i'][:500], label='I')
plt.plot(stream_data['q'][:500], label='Q')
plt.legend()
plt.title("Streamed IQ Data")
plt.xlabel("Sample")
plt.show()

In [ ]:
config = QickConfig(reps=1, soft_avgs=100)

config.readout_channel(ch=0, freq=100e6, gain=0.3, length=1024)
config.pulse(ch=0, style="gaussian", length=200, gain=0.5)

config.enable_avg(True)  # On-FPGA averaging

soc.run(config)
i_avg, q_avg = soc.get_readout()

print(f"Averaged I: {np.mean(i_avg):.3f}")
print(f"Averaged Q: {np.mean(q_avg):.3f}")

In [ ]:
# Compare streaming vs buffered readout

# Streaming mode
config_stream = QickConfig(reps=1, soft_avgs=1)
config_stream.stream_enable(True)
config_stream.stream_channel(ch=0, decimation=1)
config_stream.pulse(ch=0, style="const", length=500, gain=0.5)
soc.run(config_stream)
stream = soc.get_stream()

# Buffered mode
config_buffer = QickConfig(reps=1, soft_avgs=1)
config_buffer.readout_channel(ch=0, freq=100e6, gain=0.3, length=500)
config_buffer.pulse(ch=0, style="const", length=500, gain=0.5)
soc.run(config_buffer)
i_buf, q_buf = soc.get_readout()

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(stream['i'][:500])
plt.title("Streaming Mode")
plt.subplot(1,2,2)
plt.plot(i_buf)
plt.title("Buffered Mode")
plt.show()